<a href="https://colab.research.google.com/github/NourHassan5678/Assignments/blob/main/work/notebooks/w03_data-contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content item (page)**

**Table(s):** primarily the real warehouse now — `hf://datasets/FlyRank/internship-warehouse`, specifically `fact_content_daily_performance` (daily grain, aggregated up to content-item-per-month below) for the month-2026-03 partition.

**Time window:** `month = '2026-03'`, one full calendar month, drawn from the daily fact table's own `report_date` column.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**What I'd predict/rank (label or proxy):** `ctr_gap` — a page's CTR minus its own position tier's median CTR . A more negative gap means the page under-captures clicks relative to its tier peers, i.e. a bigger review opportunity. also `ctr_improved` (did CTR rise between `prev_30d` and `last_30d`).

**Context (IDs — never features):** `content_id`, `client_id`. Both are pseudonyms, used only for joining, grouping, and splitting (e.g. client-grouped validation later) — never fed to a model.

**Feature (knowable before the review decision):** `avg_position`, `position_tier`, `impressions_90d`, `sessions_90d`, `engagement_rate`, `scroll_rate`, `content_age_days`, `word_count`, `content_type`, `main_intent`, `freshness_tier` — all measured over the already-completed 90-day window, so none of them reach into the future relative to the review moment.

**Label / proxy inputs (never *also* a feature for the same target):** `ctr` and `position_tier` are exactly what `ctr_gap` is built FROM. `position_tier` is still safe to use as a feature (it's an input, not the label). Raw `ctr`, though, is not — the moment the target is `ctr_gap`, re-including `ctr` as a feature turns it back into the label wearing a disguise. Section 3's trap shows exactly why. `clicks_last_30d` / `impressions_last_30d` are the same story for the stretch label `ctr_improved`.

**Excluded — with why:**
- `trend_direction` and `trend_pct` — excluded outright. Per the flyrank-data skill, `trend_direction` is computed from `trend_pct`, and both summarize the same kind of before/after window comparison my own stretch label (`ctr_improved`) already measures. Using them as features would leak a close cousin of the label back in.
- `provider_used` (71.5% missing) and `model_used` (19.1% missing) — excluded for sparsity, not safety; too little coverage to trust as a feature at this stage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
import os
import requests
from pathlib import Path
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
HEADERS = {"Authorization": f"Bearer {HF_TOKEN}"}
DATASET = "FlyRank/internship-warehouse"
BASE = "https://datasets-server.huggingface.co"
TABLE = "fact_content_daily_performance"
MONTH_START = "2026-03-01"
MONTH_END = "2026-03-31"

parquet_resp = requests.get(
    f"{BASE}/parquet", params={"dataset": DATASET, "config": TABLE}, headers=HEADERS
)
parquet_resp.raise_for_status()
files = sorted(parquet_resp.json().get("parquet_files", []), key=lambda f: f["filename"])
print(f"{len(files)} parquet shard(s) found for {TABLE}.")

LOCAL_DIR = Path(f"flyrank_{TABLE}")
LOCAL_DIR.mkdir(exist_ok=True)

def download(f):
    out_path = LOCAL_DIR / f["filename"].replace("/", "__")
    if not out_path.exists():
        r = requests.get(f["url"], headers=HEADERS, stream=True)
        r.raise_for_status()
        with open(out_path, "wb") as fh:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                fh.write(chunk)
    print("Downloaded:", out_path.name, f"({out_path.stat().st_size / 1e6:.1f} MB)")
    return out_path

import duckdb
con = duckdb.connect()

target = next((f for f in files if f["filename"].endswith("0014.parquet")), None)
used_fallback = True
if target is not None:
    path = download(target)
    span = con.sql(f"SELECT MIN(report_date) a, MAX(report_date) b FROM '{path}'").df()
    a, b = str(span["a"][0])[:10], str(span["b"][0])[:10]
    print(f"\n0014.parquet actual date range: {a} to {b}")
    if a >= MONTH_START and b <= MONTH_END:
        print("Confirmed -- this shard IS March 2026. Using it alone.")
        LOCAL_GLOB = str(path)
        used_fallback = False
    else:
        print("Date range does NOT match March 2026 -- falling back to downloading everything.")

if used_fallback:
    print("\nDownloading all shards and filtering by report_date instead (slower, always correct).")
    for f in files:
        download(f)
    LOCAL_GLOB = str(LOCAL_DIR / "*.parquet")

print("\nLOCAL_GLOB:", LOCAL_GLOB)

desc = con.sql(f"DESCRIBE SELECT * FROM '{LOCAL_GLOB}' LIMIT 0").df()
print("Columns:", list(desc["column_name"]))

18 parquet shard(s) found for fact_content_daily_performance.
Downloaded: 0014.parquet (124.2 MB)

0014.parquet actual date range: 2026-03-01 to 2026-03-31
Confirmed -- this shard IS March 2026. Using it alone.

LOCAL_GLOB: flyrank_fact_content_daily_performance/0014.parquet
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


### Query 1 — Grain

The raw daily fact table's grain is `report_date x client x content` — **not** one row per page. Proving Lane 4's contracted grain ("one row = one content item") means first showing the raw table is daily, then aggregating up to content-item-per-month, then re-running the grain check on *that* — this is the actual proof, not an assumption.

In [ ]:
# Step 1: raw grain, restricted to March -- expect MANY rows per
# content_hash_id (about one per day).
raw_grain = con.sql(f"""
    SELECT content_hash_id, COUNT(*) AS c
    FROM '{LOCAL_GLOB}'
    WHERE report_date >= DATE '{MONTH_START}' AND report_date <= DATE '{MONTH_END}'
    GROUP BY content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()
print("Raw daily grain, March only:")
print(raw_grain)

# Step 2: aggregate to Lane 4's real unit of analysis -- one row per content
# item for the month -- using the REAL column names confirmed from your file.
con.sql(f"""
    CREATE OR REPLACE TABLE monthly_agg AS
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions)                   AS impressions_month,
        SUM(gsc_clicks)                        AS clicks_month,
        AVG(NULLIF(gsc_avg_position, 0))       AS avg_position_month,
        SUM(ga4_sessions)                      AS sessions_month,
        SUM(ga4_engaged_sessions)              AS engaged_sessions_month,
        BOOL_OR(ga4_data_available)            AS any_ga4_available,
        MIN(report_date)                       AS min_date,
        MAX(report_date)                       AS max_date
    FROM '{LOCAL_GLOB}'
    WHERE report_date >= DATE '{MONTH_START}' AND report_date <= DATE '{MONTH_END}'
    GROUP BY content_hash_id, client_hash_id
""")

grain_check = con.sql("""
    SELECT content_hash_id, COUNT(*) AS c
    FROM monthly_agg GROUP BY content_hash_id HAVING c > 1
""").df()
print(f"\nGrain check on the AGGREGATED table: {len(grain_check)} rows")

Raw daily grain, March only (expect rows back -- grain is daily, not content-item):
            content_hash_id   c
0  content_b7e512995f79d5a6  31
1  content_05597932fe4da067  31
2  content_905aa32a0230694e  31
3  content_05434271b257bb68  31
4  content_d056587ff7faca0c  31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Grain check on the AGGREGATED table: 0 rows (expect 0 -- grain holds)


### Query 2 — Counts + real date span

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE lane4_slice AS
    SELECT *,
        CASE
            WHEN avg_position_month > 0  AND avg_position_month <= 3  THEN 'top_3'
            WHEN avg_position_month > 3  AND avg_position_month <= 10 THEN 'page_1'
            WHEN avg_position_month > 10 AND avg_position_month <= 20 THEN 'striking'
            WHEN avg_position_month > 20 AND avg_position_month <= 50 THEN 'page_3_5'
            ELSE 'deep'
        END AS position_tier
    FROM monthly_agg
    WHERE impressions_month >= 500
      AND avg_position_month > 0
      AND avg_position_month <= 20
""")

counts = con.sql("""
    SELECT COUNT(*) AS n_rows, MIN(min_date) AS window_start, MAX(max_date) AS window_end
    FROM lane4_slice
""").df()
print(counts)

   n_rows window_start window_end
0   50717   2026-03-01 2026-03-31


### Query 3 — Availability, filtered with `IS TRUE`

The real `ga4_data_available` flag, filtered exactly the way the skill's panel warning says to: rows before a client's GA4 start are zero-filled, not "zero engagement" — so this filter matters, not just as a style choice.

In [ ]:
avail = con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE any_ga4_available IS TRUE) AS n_available,
        COUNT(*) AS n_total
    FROM lane4_slice
""").df()
avail["pct_available"] = (avail["n_available"] / avail["n_total"] * 100).round(1)
print(avail)

   n_available  n_total  pct_available
0        31754    50717           62.6


### Five features, max — each knowable before the review decision

1. `avg_position_month` — average GSC rank measured over the already-completed month; nothing about a future review depends on it changing first.
2. `impressions_month` — a completed sum of past search exposure for the month.
3. `sessions_month` — a completed-window GA4 measurement (subject to the `any_ga4_available` gate above).
4. `engaged_sessions_month` — same reasoning as sessions; already happened by review time.
5. `position_tier` — a bucket derived purely from `avg_position_month`, knowable at exactly the same moment that is.

Deliberately **not** included: `ctr_month` (clicks/impressions) — it's the input the label (`ctr_gap`) is built from, so it goes in the trap below, not the honest feature set.

In [ ]:
lane4_df = con.sql("SELECT * FROM lane4_slice").df()

lane4_df["ctr_month"] = lane4_df["clicks_month"] / lane4_df["impressions_month"] * 100
lane4_df["expected_ctr_for_tier"] = lane4_df.groupby("position_tier")["ctr_month"].transform("median")
lane4_df["ctr_gap"] = lane4_df["ctr_month"] - lane4_df["expected_ctr_for_tier"]

ga4_cols = ["sessions_month", "engaged_sessions_month"]
n_missing = lane4_df[ga4_cols].isna().any(axis=1).sum()
lane4_df[ga4_cols] = lane4_df[ga4_cols].fillna(0)
print(f"Rows with missing GA4 sums, filled to 0: {n_missing:,} / {len(lane4_df):,}")

print(f"\nLane 4 monthly feature frame: {len(lane4_df):,} rows")
lane4_df[["content_hash_id", "position_tier", "avg_position_month",
          "impressions_month", "sessions_month", "engaged_sessions_month", "ctr_gap"]].head(8)

Rows with missing GA4 sums, filled to 0: 13,089 / 50,717

Lane 4 monthly feature frame: 50,717 rows


,content_hash_id,position_tier,avg_position_month,impressions_month,sessions_month,engaged_sessions_month,ctr_gap
0,content_cec711b02f3bbde6,page_1,4.428747,602.0,0.0,0.0,0.447170
1,content_614baf2af4330bd7,page_1,4.685335,772.0,0.0,0.0,-0.087749
2,content_755d951187fcd70a,top_3,1.854929,1858.0,0.0,0.0,0.059565
3,content_cdd114d71966c437,striking,10.400049,1888.0,0.0,0.0,-0.162075
4,content_7dbc094b799e05a4,page_1,6.155424,705.0,0.0,0.0,-0.075438
5,content_d8d69e789e933656,page_1,6.984338,826.0,0.0,0.0,-0.217282
6,content_df22bda1218f13ff,page_1,3.066796,2099.0,0.0,0.0,-0.169641
7,content_29a55b5735fc1e4e,page_1,4.709817,829.0,0.0,0.0,-0.096655


### The trap: add one label-derived column on purpose

Same trap as `w02` and `w03`'s CSV version, now proven on real warehouse data. `ctr_gap` is *defined* as `ctr_month - expected_ctr_for_tier`. Adding raw `ctr_month` back in as a "feature" hands a linear model the exact arithmetic to reconstruct the label — watch the quick score.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

feature_cols = ["avg_position_month", "impressions_month", "sessions_month", "engaged_sessions_month"]
tier_dummies = pd.get_dummies(lane4_df["position_tier"], prefix="tier")
X_honest = pd.concat([lane4_df[feature_cols].reset_index(drop=True), tier_dummies.reset_index(drop=True)], axis=1)
y = lane4_df["ctr_gap"].reset_index(drop=True)

Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.2, random_state=42)
honest_model = LinearRegression().fit(Xtr, ytr)
r2_honest = r2_score(yte, honest_model.predict(Xte))
print(f"HONEST 5-feature quick score (R^2, held-out): {r2_honest:.3f}")

# The leak, added on purpose:
X_leak = X_honest.copy()
X_leak["ctr_month"] = lane4_df["ctr_month"].reset_index(drop=True)

Xtr2, Xte2, ytr2, yte2 = train_test_split(X_leak, y, test_size=0.2, random_state=42)
leak_model = LinearRegression().fit(Xtr2, ytr2)
r2_leak = r2_score(yte2, leak_model.predict(Xte2))
print(f"LEAKED 6-feature quick score (R^2, held-out): {r2_leak:.3f}")
print(f"Jump: {r2_honest:.3f} -> {r2_leak:.3f}")

HONEST 5-feature quick score (R^2, held-out): 0.105
LEAKED 6-feature quick score (R^2, held-out): 1.000
Jump: 0.105 -> 1.000


In [ ]:
del X_leak, leak_model, r2_leak

final_r2 = r2_honest
print(f"Honest quick score carried into the modeling weeks: R^2 = {final_r2:.3f}")

Honest quick score carried into the modeling weeks: R^2 = 0.105


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** `month=2026-03` is one month of daily rows aggregated up to a monthly page-level slice. A client whose `ga4_data_start` falls mid-March shows `any_ga4_available = TRUE` for the month (the `BOOL_OR`), but their `sessions_month`/`engaged_sessions_month` sums only cover the days after that start — partial-month GA4 coverage looks the same as full-month coverage in this aggregation unless checked against `dim_clients.gsc_data_start`/`ga4_data_start` directly.

Other real limits, now confirmed rather than hypothetical:

- **GA4 availability really is unbalanced early in the panel.** Opening the actual `2025-04` shard directly showed `ga4_data_available = False` for every single row that month — not a theoretical risk, a confirmed fact about this dataset. March 2026 sits much later in the panel, so I'd expect far better GA4 coverage there, but that's an expectation, not something I've verified for that specific month.
- **History depth differs wildly per client.** A third of clients have little or no usable history at all; per-client windows (via `dim_clients`) would be more honest than one shared calendar month.
- **One month is not the full picture.** Results from `2026-03` alone need to be re-checked against other months before trusting any pattern as durable rather than a one-month artifact.